# Blocks & Streets
## Where NYC housing distress meets street danger

**Data Engineering and Visualization, Winter 2026, Milestone 1**
German International University, Faculty of Informatics and Computer Science

| | |
|---|---|
| **Housing & Development dataset** | Housing Maintenance Code Complaints and Problems (HPD), `ygpa-z7cr` (catalog **H6**) |
| **Transportation dataset** | Motor Vehicle Collisions, Crashes (NYPD), `h9gi-nx95` |
| **Integration key** | 5-digit ZIP code × calendar month (Jan 2024 to Dec 2025) |
| **Website** | Dash (Python/Plotly), deployed on Render |

### The question
NYC agencies study building conditions and street safety separately. HPD sees broken boilers, leaks and mold. DOT and NYPD see crashes.
But both are symptoms of how much care a neighborhood receives. **We ask whether the ZIP codes where housing is failing are also the
ZIP codes where streets injure people on foot and on bikes**, and whether the two burdens move together over time or simply sit in the
same places.

**Why it matters:** a household in a building with no heat that also walks its children across a dangerous arterial carries a *double burden*.
If the burdens overlap, HPD inspections and Vision Zero street redesigns could be targeted together instead of separately.

### Notebook map
0. Setup and configuration
1. Dataset selection, links and feasibility check
2. Loading (filtered sample through the SoQL API, with caching)
3. Exploratory data analysis (raw)
4. Pre-integration cleaning (housing, then crashes)
5. Integration (ZIP × month panel)
6. Post-integration cleaning
7. Research questions (10 questions, 10 visualizations)
8. Export for the website
9. Limitations, impact, contributions, prompts used

## 0. Setup and configuration
All tunable choices live in one cell so the pipeline is reproducible. The study window is **two full calendar years** (2024 and 2025):
two years give a year-over-year comparison (RQ10) and two complete seasonal cycles (RQ3, RQ4), while keeping the housing table small
enough for a free Colab runtime.

In [1]:
!pip install -q pandas plotly scipy pyarrow   # uncomment on a fresh Colab runtime if needed

import os, time, json, warnings
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# ---------- study configuration ----------
HOUSING_ID = "ygpa-z7cr"          # Housing Maintenance Code Complaints and Problems
CRASH_ID = "h9gi-nx95"            # Motor Vehicle Collisions - Crashes
STUDY_START = "2024-01-01"        # inclusive
STUDY_END = "2025-12-31"          # inclusive
PAGE_SIZE = 500_000               # rows per API page
APP_TOKEN = os.environ.get("SOCRATA_APP_TOKEN")  # optional, raises the API rate limit

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"        # cached monthly downloads (parquet)
PROC_DIR = DATA_DIR / "processed" # small aggregated tables used by the website
for d in (RAW_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASET_LINKS = {
    "Housing Maintenance Code Complaints and Problems": f"https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Complaints-and-Problems/{HOUSING_ID}",
    "Motor Vehicle Collisions - Crashes": f"https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/{CRASH_ID}",
}

# ---------- shared visual style (validated categorical palette, fixed order) ----------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRAY = "#9a9994"
BORO_ORDER = ["MANHATTAN", "BRONX", "BROOKLYN", "QUEENS", "STATEN ISLAND"]
BORO_COLOR = dict(zip(BORO_ORDER, PALETTE[:5]))         # colour follows the borough, never its rank
BORO_SYMBOL = dict(zip(BORO_ORDER, ["circle", "square", "diamond", "triangle-up", "x"]))  # secondary encoding

def style(fig, title, x=None, y=None, height=460):
    """Apply one consistent, recessive chart style and clear axis labels."""
    fig.update_layout(
        title=dict(text=title, x=0.01, font=dict(size=16)),
        template="plotly_white", height=height, font=dict(family="Inter, Arial", size=12, color="#0b0b0b"),
        margin=dict(l=60, r=30, t=70, b=55), legend=dict(title=None, orientation="h", y=-0.18),
        hoverlabel=dict(bgcolor="white"),
    )
    fig.update_xaxes(showgrid=True, gridcolor="#ecebe8", linecolor="#c3c2b7", title=x)
    fig.update_yaxes(showgrid=True, gridcolor="#ecebe8", linecolor="#c3c2b7", title=y)
    return fig

print("pandas", pd.__version__, "| plotly", __import__("plotly").__version__)

pandas 2.2.3 | plotly 5.24.1


## 1. Dataset selection, links and feasibility check

### Why these two datasets
| Criterion | HPD Complaints & Problems | NYPD Motor Vehicle Collisions |
|---|---|---|
| Domain | Housing & Development (catalog H6) | Transportation (street safety; the project brief lists crashes as a transportation signal) |
| Unit of observation | One *problem* inside a complaint (e.g. "no heat, apartment 3B") | One crash event |
| Geography | Borough, ZIP (`post_code`), lat/lon, building ID, NTA | Borough, ZIP, lat/lon, street names |
| Time | `received_date` (daily) | `crash_date` (daily) + `crash_time` (minute) |
| Rich categories | major/minor category, urgency type, status | contributing factor, vehicle type, victim type (pedestrian, cyclist, motorist) |
| Columns | 30+ (>= 9 required) | 29 (>= 9 required) |

**Why not a row-level join?** A complaint and a crash are different events. Matching "the crash outside the building with the leak"
would be spurious precision. Both datasets carry a **5-digit ZIP**, the finest geography they *both* report reliably, so we aggregate
both to **ZIP × month** and join there.

**Why ZIP × month?** ZIP keeps neighborhood contrast (roughly 180 residential ZIPs) while giving each cell enough events for stable
rates. Month keeps the seasonal signal (heat season, summer crash peak) that a yearly table would erase. Daily or block-level cells would
be mostly zeros.

### Feasibility check (the crash dataset is not the catalog's T4 SIRTA table, so we document it)
The cell below queries both APIs for total row counts and the full column list, then prints the attributes used for integration.

In [2]:
API = "https://data.cityofnewyork.us/resource/{}.csv"
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else None

def soql_get(dataset_id: str, params: dict, retries: int = 4) -> pd.DataFrame:
    """Call the Socrata SoQL API and return a DataFrame of strings. Retries with exponential back-off."""
    url = API.format(dataset_id) + "?" + urlencode(params)
    for attempt in range(retries):
        try:
            return pd.read_csv(url, dtype=str, low_memory=False, storage_options=HEADERS)
        except Exception as err:  # network errors, 429 rate limits, 5xx
            wait = 5 * 2 ** attempt
            print(f"  request failed ({type(err).__name__}: {str(err)[:80]}), retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"API request failed after {retries} attempts: {url}")

def dataset_profile(dataset_id: str, cache: Path) -> dict:
    """Total rows + all column names for a dataset. Cached so the notebook re-runs offline."""
    if cache.exists():
        return json.loads(cache.read_text())
    cols = list(soql_get(dataset_id, {"$limit": 5}).columns)
    total = int(soql_get(dataset_id, {"$select": "count(*) AS n"})["n"].iloc[0])
    prof = {"dataset_id": dataset_id, "total_rows": total, "n_columns": len(cols), "columns": cols}
    cache.write_text(json.dumps(prof))
    return prof

housing_profile = dataset_profile(HOUSING_ID, RAW_DIR / "profile_housing.json")
crash_profile = dataset_profile(CRASH_ID, RAW_DIR / "profile_crash.json")

overview = pd.DataFrame([
    {"dataset": "HPD Complaints & Problems", "id": HOUSING_ID, "total rows (full history)": housing_profile["total_rows"],
     "columns": housing_profile["n_columns"], "geographic attributes": "borough, post_code (ZIP), latitude/longitude, building_id, nta",
     "temporal attributes": "received_date, problem_status_date"},
    {"dataset": "NYPD Motor Vehicle Collisions", "id": CRASH_ID, "total rows (full history)": crash_profile["total_rows"],
     "columns": crash_profile["n_columns"], "geographic attributes": "borough, zip_code, latitude/longitude, street names",
     "temporal attributes": "crash_date, crash_time"},
])
display(overview)
print("\nHousing columns:", housing_profile["columns"])
print("\nCrash columns:", crash_profile["columns"])

,dataset,id,total rows (full history),columns,geographic attributes,temporal attributes
0,HPD Complaints & Problems,ygpa-z7cr,16445945,33,"borough, post_code (ZIP), latitude/longitude, ...","received_date, problem_status_date"
1,NYPD Motor Vehicle Collisions,h9gi-nx95,2303506,29,"borough, zip_code, latitude/longitude, street ...","crash_date, crash_time"



Housing columns: ['received_date', 'problem_id', 'complaint_id', 'building_id', 'borough', 'house_number', 'street_name', 'post_code', 'block', 'lot', 'apartment', 'community_board', 'unit_type', 'space_type', 'type', 'major_category', 'minor_category', 'problem_code', 'complaint_status', 'complaint_status_date', 'problem_status', 'problem_status_date', 'status_description', 'problem_duplicate_flag', 'complaint_anonymous_flag', 'unique_key', 'latitude', 'longitude', 'council_district', 'census_tract', 'bin', 'bbl', 'nta']

Crash columns: ['crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'location', 'on_street_name', 'off_street_name', 'cross_street_name', 'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed', 'contributing_factor_vehicle_1', 'contributing_factor_vehicle_2', 'con

**Feasibility verdict.** Both tables exceed the 9-column minimum, share ZIP, borough and date, and are large enough that every ZIP × month
cell has events. The expected integration is a **many-to-one aggregation on both sides** into one shared ZIP × month panel. The sizes are
very different (millions of housing problems vs hundreds of thousands of crashes), which is exactly why we aggregate instead of joining rows.

## 2. Loading: a justified filtered sample
The housing table holds well over ten million problems since 2019, which would crash a free notebook if downloaded in full.
**Sampling choice:** we keep *every* record inside the study window (2024 to 2025) instead of a random sample, because random sampling would
break building-level counts (RQ1, RQ5) and repair-time statistics (RQ7). We download **one month at a time**, request only the columns
we use (`$select`), page through results with a stable `$order=:id`, and cache each month as parquet so re-runs are instant.

Column names can differ between the API and the CSV export, so we first map whatever names the API returns onto one canonical schema.

In [3]:
# canonical name -> candidate API names (first match wins). Keeps the pipeline robust to schema renames.
HOUSING_SCHEMA = {
    "problem_id": ["problem_id", "problemid"],
    "complaint_id": ["complaint_id", "complaintid"],
    "building_id": ["building_id", "buildingid"],
    "borough": ["borough", "boro"],
    "zip": ["post_code", "postcode", "zip", "zip_code", "zipcode"],
    "received_date": ["received_date", "receiveddate"],
    "urgency": ["type", "problem_type"],
    "major_category": ["major_category", "majorcategory"],
    "minor_category": ["minor_category", "minorcategory"],
    "problem_status": ["problem_status", "status"],
    "problem_status_date": ["problem_status_date", "status_date", "statusdate"],
    "complaint_status": ["complaint_status"],
    "duplicate_flag": ["problem_duplicate_flag", "duplicate_flag"],
    "unit_type": ["unit_type", "unittype"],
    "space_type": ["space_type", "spacetype"],
    "latitude": ["latitude"],
    "longitude": ["longitude"],
    "nta": ["nta"],
}
CRASH_SCHEMA = {
    "collision_id": ["collision_id"],
    "crash_date": ["crash_date"],
    "crash_time": ["crash_time"],
    "borough": ["borough"],
    "zip": ["zip_code", "zip"],
    "latitude": ["latitude"],
    "longitude": ["longitude"],
    "on_street_name": ["on_street_name"],
    "persons_injured": ["number_of_persons_injured"],
    "persons_killed": ["number_of_persons_killed"],
    "ped_injured": ["number_of_pedestrians_injured"],
    "ped_killed": ["number_of_pedestrians_killed"],
    "cyc_injured": ["number_of_cyclist_injured"],
    "cyc_killed": ["number_of_cyclist_killed"],
    "mot_injured": ["number_of_motorist_injured"],
    "mot_killed": ["number_of_motorist_killed"],
    "factor_1": ["contributing_factor_vehicle_1"],
    "factor_2": ["contributing_factor_vehicle_2"],
    "vehicle_type_1": ["vehicle_type_code1", "vehicle_type_code_1"],
    "vehicle_type_2": ["vehicle_type_code2", "vehicle_type_code_2"],
}
REQUIRED_HOUSING = ["problem_id", "building_id", "borough", "zip", "received_date", "major_category", "urgency"]
REQUIRED_CRASH = ["collision_id", "crash_date", "crash_time", "borough", "zip", "latitude", "longitude", "persons_injured",
                  "ped_injured", "cyc_injured", "factor_1", "vehicle_type_1"]

def resolve_schema(api_columns, schema, required, name):
    """Return {api_name: canonical_name} for every canonical column the API offers; fail loudly if a required one is missing."""
    lower = {c.lower(): c for c in api_columns}
    mapping = {}
    for canon, candidates in schema.items():
        hit = next((lower[c] for c in candidates if c in lower), None)
        if hit:
            mapping[hit] = canon
    missing = [c for c in required if c not in mapping.values()]
    assert not missing, f"{name}: required columns not found in API schema: {missing}. Check df.columns and update the schema map."
    return mapping

HOUSING_MAP = resolve_schema(housing_profile["columns"], HOUSING_SCHEMA, REQUIRED_HOUSING, "housing")
CRASH_MAP = resolve_schema(crash_profile["columns"], CRASH_SCHEMA, REQUIRED_CRASH, "crash")
pd.DataFrame({"housing (API -> canonical)": pd.Series(HOUSING_MAP), }).T

,problem_id,complaint_id,building_id,borough,post_code,received_date,type,major_category,minor_category,problem_status,problem_status_date,complaint_status,problem_duplicate_flag,unit_type,space_type,latitude,longitude,nta
housing (API -> canonical),problem_id,complaint_id,building_id,borough,zip,received_date,urgency,major_category,minor_category,problem_status,problem_status_date,complaint_status,duplicate_flag,unit_type,space_type,latitude,longitude,nta


In [5]:
def month_starts(start, end):
    return pd.date_range(pd.Timestamp(start).replace(day=1), pd.Timestamp(end), freq="MS")

def download_month(dataset_id, mapping, date_api_col, month_start, prefix):
    """Download one calendar month (all pages) with only the mapped columns; cache to parquet."""
    cache = RAW_DIR / f"{prefix}_{month_start:%Y_%m}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    nxt = month_start + pd.offsets.MonthBegin(1)
    where = f"{date_api_col} >= '{month_start:%Y-%m-%dT00:00:00}' AND {date_api_col} < '{nxt:%Y-%m-%dT00:00:00}'"
    pages, offset = [], 0
    while True:
        page = soql_get(dataset_id, {"$select": ",".join(mapping.keys()), "$where": where,
                                     "$order": ":id", "$limit": PAGE_SIZE, "$offset": offset})
        pages.append(page)
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
    df = pd.concat(pages, ignore_index=True)
    df.to_parquet(cache, index=False)
    return df

def load_dataset(dataset_id, mapping, canonical_date, prefix):
    date_api_col = next(k for k, v in mapping.items() if v == canonical_date)
    frames = []
    for m in month_starts(STUDY_START, STUDY_END):
        t0 = time.time()
        frames.append(download_month(dataset_id, mapping, date_api_col, m, prefix))
        print(f"{prefix} {m:%Y-%m}: {len(frames[-1]):>9,} rows ({time.time() - t0:.1f}s)")
    df = pd.concat(frames, ignore_index=True)[list(mapping)].rename(columns=mapping)
    # memory: low-cardinality text columns become categoricals (millions of repeated strings -> integer codes)
    for c in df.columns:
        if df[c].nunique(dropna=True) < 0.01 * len(df):
            df[c] = df[c].astype("category")
    return df

housing_raw = load_dataset(HOUSING_ID, HOUSING_MAP, "received_date", "housing")
crash_raw = load_dataset(CRASH_ID, CRASH_MAP, "crash_date", "crash")
print(f"\nHousing sample: {housing_raw.shape[0]:,} rows x {housing_raw.shape[1]} columns")
print(f"Crash sample:   {crash_raw.shape[0]:,} rows x {crash_raw.shape[1]} columns")

housing 2024-01:   108,269 rows (0.4s)
housing 2024-02:    72,452 rows (0.2s)
housing 2024-03:    71,768 rows (0.2s)
housing 2024-04:    61,810 rows (0.1s)
housing 2024-05:    50,284 rows (0.1s)
housing 2024-06:    46,110 rows (0.1s)
housing 2024-07:    53,110 rows (0.1s)
housing 2024-08:    53,736 rows (0.1s)
housing 2024-09:    49,533 rows (0.1s)
housing 2024-10:    66,485 rows (0.2s)
housing 2024-11:    81,000 rows (0.2s)
housing 2024-12:   108,612 rows (0.3s)
housing 2025-01:   117,276 rows (0.3s)
housing 2025-02:    80,339 rows (0.2s)
housing 2025-03:    70,095 rows (0.2s)
housing 2025-04:    58,817 rows (0.2s)
housing 2025-05:    48,985 rows (0.1s)
housing 2025-06:    46,989 rows (0.1s)
housing 2025-07:    52,231 rows (0.1s)
housing 2025-08:    46,964 rows (0.1s)
housing 2025-09:    48,773 rows (0.1s)
housing 2025-10:    81,378 rows (0.3s)
housing 2025-11:    92,527 rows (0.2s)
housing 2025-12:   111,906 rows (18.2s)
crash 2024-01:     7,542 rows (5.8s)
crash 2024-02:     6,841 r

## 3. Exploratory data analysis (raw data)
Goal: understand structure and spot problems *before* cleaning. Everything is still a string, exactly as the API delivered it.

In [6]:
def profile_table(df: pd.DataFrame) -> pd.DataFrame:
    """Per-column profile: missing share, distinct values and an example value."""
    as_str = df.astype("string")
    blank = as_str.isna() | as_str.apply(lambda s: s.str.strip().eq(""))
    return pd.DataFrame({
        "missing_%": (blank.mean() * 100).round(2),
        "distinct": df.nunique(dropna=True),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    }).sort_values("missing_%", ascending=False)

print("HOUSING raw profile"); display(profile_table(housing_raw))
print("CRASH raw profile"); display(profile_table(crash_raw))

HOUSING raw profile


,missing_%,distinct,example
zip,0.08,204,10039
latitude,0.02,71322,40.827575000000003
nta,0.02,217,Harlem (North)
longitude,0.02,68791,-73.938443000000007
complaint_id,0.0,1024874,12624342
problem_id,0.0,1679449,24964398
urgency,0.0,3,NON EMERGENCY
building_id,0.0,83223,4197
borough,0.0,5,MANHATTAN
received_date,0.0,1007588,2024-01-01T00:01:23.000


CRASH raw profile


,missing_%,distinct,example
vehicle_type_2,34.01,655,Station Wagon/Sport Utility Vehicle
on_street_name,29.53,8141,EAST 21 STREET
zip,24.32,218,10010
factor_2,24.27,55,Unspecified
borough,18.75,5,MANHATTAN
latitude,4.42,49338,40.737545
longitude,4.42,36894,-73.98409
vehicle_type_1,1.73,618,E-Scooter
factor_1,0.74,56,Driver Inexperience
crash_time,0.0,1440,11:00


In [7]:
miss = pd.concat([
    profile_table(housing_raw)["missing_%"].rename("value").to_frame().assign(dataset="Housing problems"),
    profile_table(crash_raw)["missing_%"].rename("value").to_frame().assign(dataset="Crashes"),
]).reset_index(names="column")
fig = px.bar(miss.sort_values("value"), x="value", y="column", facet_col="dataset", orientation="h",
             color_discrete_sequence=[PALETTE[0]], facet_col_spacing=0.18)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(matches=None, showticklabels=True)
style(fig, "Missing values by column (raw)", x="% of rows missing", height=620).show()

**What the missingness tells us**
* Crashes: `borough` and `zip` are missing for a large share of crashes (NYPD geocodes many crashes only to coordinates or street names).
  Because ZIP is our join key, dropping these rows would throw away real crashes and bias against highways and arterials.
  We will **impute ZIP from coordinates** (Section 4.2) rather than drop.
* Housing: almost every problem has a ZIP and building ID, so ZIP gaps can be filled deterministically from the same building's other records.
* Secondary fields (`factor_2`, `vehicle_type_2`) are missing when only one vehicle was involved. That is *structural* missingness, not error.

In [8]:
# Quick look at categorical structure and volume over time (raw strings, so dates parsed only for this plot)
h_month = pd.to_datetime(housing_raw["received_date"].astype("string"), errors="coerce").dt.to_period("M").value_counts().sort_index()
c_month = pd.to_datetime(crash_raw["crash_date"].astype("string"), errors="coerce").dt.to_period("M").value_counts().sort_index()
vol = pd.concat([h_month.rename("Housing problems"), c_month.rename("Crashes")], axis=1)
vol.index = vol.index.to_timestamp()
fig = go.Figure()
for i, col in enumerate(vol.columns):
    fig.add_scatter(x=vol.index, y=vol[col], name=col, mode="lines", line=dict(width=2, color=PALETTE[i]))
fig.update_yaxes(type="log")
style(fig, "Raw monthly volume: housing problems vs crashes (log scale, different magnitudes)", y="records per month (log)").show()

top_major = housing_raw["major_category"].astype("string").str.upper().str.strip().value_counts().head(15)
fig = px.bar(top_major[::-1], orientation="h", color_discrete_sequence=[PALETTE[0]])
style(fig, "Housing problems by major category (raw, top 15)", x="problems", y=None).update_layout(showlegend=False).show()

top_factor = crash_raw["factor_1"].astype("string").value_counts().head(15)
fig = px.bar(top_factor[::-1], orientation="h", color_discrete_sequence=[PALETTE[1]])
style(fig, "Crashes by primary contributing factor (raw, top 15)", x="crashes", y=None).update_layout(showlegend=False).show()

In [9]:
# Inconsistencies worth fixing: spelling variants, mixed case, invalid ZIPs, coordinates at (0,0)
print("Distinct raw vehicle type strings:", crash_raw["vehicle_type_1"].nunique())
print(crash_raw["vehicle_type_1"].astype("string").value_counts().head(25).to_string())
print("\nBorough spellings (housing):", housing_raw["borough"].astype("string").value_counts(dropna=False).to_dict())
print("Borough spellings (crashes):", crash_raw["borough"].astype("string").value_counts(dropna=False).to_dict())
bad_zip_h = ~housing_raw["zip"].astype("string").str.fullmatch(r"\d{5}", na=False)
bad_zip_c = ~crash_raw["zip"].astype("string").str.fullmatch(r"\d{5}", na=False)
print(f"\nNon 5-digit ZIP values: housing {bad_zip_h.mean():.1%}, crashes {bad_zip_c.mean():.1%}")
lat = pd.to_numeric(crash_raw["latitude"].astype("string"), errors="coerce")
print(f"Crash rows with latitude 0 or missing: {((lat == 0) | lat.isna()).mean():.1%}")
inj = pd.to_numeric(crash_raw["persons_injured"].astype("string"), errors="coerce")
print("\nPersons injured per crash (raw):"); print(inj.describe().round(2).to_string())

Distinct raw vehicle type strings: 618
vehicle_type_1
Sedan                                  79810
Station Wagon/Sport Utility Vehicle    60207
Taxi                                    5245
Pick-up Truck                           4361
Bike                                    3634
Bus                                     3431
Box Truck                               2727
Motorcycle                              1927
Moped                                   1596
E-Bike                                  1207
Tractor Truck Diesel                    1168
Van                                     1067
Ambulance                                882
E-Scooter                                698
Dump                                     591
Convertible                              518
PK                                       462
Garbage or Refuse                        428
Standing Scooter                         365
Flat Bed                                 299
Carry All                                226
T

**EDA takeaways that drive cleaning**
1. Vehicle types are free text with hundreds of spellings (`Sedan`, `SEDAN`, `4 dr sedan`, `Station Wagon/Sport Utility Vehicle`). We need a grouping.
2. Boroughs appear in mixed forms; some crashes have none.
3. A share of crash coordinates are `0` or missing, which are geocoding failures, not crashes in the Atlantic.
4. Injury counts are **zero-inflated** (most crashes injure nobody). Classic IQR outlier rules will misbehave here (shown in 4.2).
5. The two tables differ in volume by an order of magnitude, confirming the need to aggregate before integrating.

## 4. Pre-integration cleaning
Shared helpers first: borough names, ZIP validation and the NYC bounding box are standardized **the same way in both datasets**,
otherwise the join would silently fail on `"Brooklyn"` vs `"BROOKLYN"` or `11201` vs `"11201.0"`.

In [10]:
BORO_ALIASES = {"1": "MANHATTAN", "MN": "MANHATTAN", "NEW YORK": "MANHATTAN", "2": "BRONX", "BX": "BRONX", "THE BRONX": "BRONX",
                "3": "BROOKLYN", "BK": "BROOKLYN", "KINGS": "BROOKLYN", "4": "QUEENS", "QN": "QUEENS",
                "5": "STATEN ISLAND", "SI": "STATEN ISLAND", "RICHMOND": "STATEN ISLAND"}
# First three ZIP digits -> borough (USPS sectional centers that serve NYC)
ZIP_PREFIX_BORO = {"100": "MANHATTAN", "101": "MANHATTAN", "102": "MANHATTAN", "103": "STATEN ISLAND", "104": "BRONX",
                   "112": "BROOKLYN", "110": "QUEENS", "111": "QUEENS", "113": "QUEENS", "114": "QUEENS", "116": "QUEENS"}
NYC_LAT, NYC_LON = (40.49, 40.92), (-74.27, -73.68)

def std_text(s: pd.Series) -> pd.Series:
    """Upper-case, trim, collapse internal whitespace; empty strings become missing."""
    s = s.astype("string").str.upper().str.strip().str.replace(r"\s+", " ", regex=True)
    return s.mask(s.eq("") | s.isin(["NAN", "NONE", "NULL", "N/A"]))

def std_borough(s: pd.Series) -> pd.Series:
    s = std_text(s).replace(BORO_ALIASES)
    return s.where(s.isin(BORO_ORDER))

def std_zip(s: pd.Series) -> pd.Series:
    """Keep the first five digits ('11201-1234' and '11201.0' both -> '11201'); reject non-NYC prefixes."""
    z = s.astype("string").str.strip().str.extract(r"^(\d{5})", expand=False)
    return z.where(z.str[:3].isin(ZIP_PREFIX_BORO.keys()))

def clean_coords(df: pd.DataFrame) -> pd.DataFrame:
    """Numeric lat/lon; anything outside the NYC bounding box (including 0,0) becomes missing."""
    lat = pd.to_numeric(df["latitude"].astype("string"), errors="coerce")
    lon = pd.to_numeric(df["longitude"].astype("string"), errors="coerce")
    ok = lat.between(*NYC_LAT) & lon.between(*NYC_LON)
    df["latitude"], df["longitude"] = lat.where(ok), lon.where(ok)
    return df

def to_xy_m(lat, lon):
    """Equirectangular projection to metres (accurate to <1% across NYC) so KD-tree distances are in metres."""
    return np.column_stack([(lon + 74.0) * 111_320 * np.cos(np.radians(40.7)), (lat - 40.5) * 110_540])

def impute_zip_from_coords(df: pd.DataFrame, max_dist_m: float = 200.0, label: str = "") -> pd.DataFrame:
    """Nearest-neighbour ZIP imputation: a record with coordinates but no ZIP takes the ZIP of the closest record
    (same dataset) that has one, if that neighbour is within `max_dist_m`. 200 m is about two short NYC blocks:
    close enough that crossing a ZIP boundary is rare, far enough to cover most geocoded points."""
    known = df["zip"].notna() & df["latitude"].notna()
    target = df["zip"].isna() & df["latitude"].notna()
    if target.sum() == 0 or known.sum() == 0:
        return df
    ref = df.loc[known, ["latitude", "longitude", "zip"]]
    if len(ref) > 400_000:  # a dense reference sample is enough and keeps memory low
        ref = ref.sample(400_000, random_state=42)
    tree = cKDTree(to_xy_m(ref["latitude"].to_numpy(), ref["longitude"].to_numpy()))
    dist, idx = tree.query(to_xy_m(df.loc[target, "latitude"].to_numpy(), df.loc[target, "longitude"].to_numpy()), k=1)
    filled = pd.Series(ref["zip"].to_numpy()[idx], index=df.index[target]).where(dist <= max_dist_m)
    df.loc[target, "zip"] = filled
    df.loc[target, "zip_imputed"] = filled.notna()
    print(f"{label}: ZIP imputed from coordinates for {filled.notna().sum():,} of {target.sum():,} candidates "
          f"(median neighbour distance {np.median(dist):.0f} m)")
    return df

def reconcile_borough(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Borough per ZIP = the most common reported borough for that ZIP (data-driven), falling back to the ZIP prefix.
    Fixes missing boroughs and flags records whose reported borough contradicts their ZIP."""
    mode = df.dropna(subset=["zip", "borough"]).groupby("zip")["borough"].agg(lambda s: s.mode().iat[0])
    implied = df["zip"].map(mode).fillna(df["zip"].str[:3].map(ZIP_PREFIX_BORO))
    conflict = df["borough"].notna() & implied.notna() & (df["borough"] != implied)
    print(f"{label}: borough filled for {(df['borough'].isna() & implied.notna()).sum():,} rows; "
          f"{conflict.sum():,} rows ({conflict.mean():.2%}) contradicted their ZIP and were corrected")
    df["borough"] = implied.where(implied.notna(), df["borough"])
    return df

### 4.1 Housing problems
| Issue | Decision | Why |
|---|---|---|
| Exact duplicate rows / repeated `problem_id` | Drop, keep the latest status | One problem should count once; the latest status is the most accurate |
| `problem_duplicate_flag = Y` | Keep the row, exclude it from distress counts | HPD flags a problem that repeats an already-open one; counting both double counts one broken boiler |
| Missing / malformed ZIP | Impute from the same `building_id` first, then from coordinates | A building's ZIP never changes, so building-based imputation is exact |
| Borough spelling and ZIP conflicts | Standardize, then reconcile to the ZIP's dominant borough | The join key must be consistent across datasets |
| Dates outside the window or unparseable | Drop | They cannot be placed in a month |
| Urgency `type` free text | Map to 4 groups; *urgent* = emergency, immediate emergency or hazardous | Gives a severity measure independent of volume |
| Repair time negative | Set to missing | Physically impossible, a data-entry error |
| Repair time extreme | Detect with IQR on log days; **cap at 365 days** for averages, keep the record | Long waits are real but a handful of decade-old administrative closures would dominate means |

In [11]:
def clean_housing(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    log = {"raw rows": len(df)}

    # 1) duplicates: exact rows, then repeated problem_id (keep latest status update)
    df = df.drop_duplicates()
    log["after exact duplicates"] = len(df)
    df["problem_status_date"] = pd.to_datetime(df["problem_status_date"].astype("string"), errors="coerce") if "problem_status_date" in df else pd.NaT
    df = df.sort_values("problem_status_date").drop_duplicates("problem_id", keep="last")
    log["after duplicate problem_id"] = len(df)

    # 2) standardize formats
    df["received_date"] = pd.to_datetime(df["received_date"].astype("string"), errors="coerce")
    for col in ["major_category", "minor_category", "urgency", "problem_status", "complaint_status",
                "duplicate_flag", "unit_type", "space_type"]:
        if col in df:
            df[col] = std_text(df[col])
    df["major_category"] = (df["major_category"].str.replace(r"\s*/\s*", "/", regex=True)
                            .replace({"HEATING": "HEAT/HOT WATER", "HEAT": "HEAT/HOT WATER", "NONCONST": "GENERAL"})
                            .fillna("UNKNOWN"))
    df["borough"] = std_borough(df["borough"])
    df["zip"] = std_zip(df["zip"])
    df = clean_coords(df)

    # 3) window filter (dates are the temporal join key)
    in_window = df["received_date"].between(STUDY_START, pd.Timestamp(STUDY_END) + pd.Timedelta(days=1), inclusive="left")
    df = df[in_window].copy()
    log["after date window"] = len(df)

    # 4) ZIP imputation: building first (exact), then coordinates (approximate)
    df["zip_imputed"] = False
    # most frequent ZIP per building, vectorised (a per-group lambda would be far too slow on ~1M buildings)
    bld_zip = (df.dropna(subset=["zip"]).groupby(["building_id", "zip"]).size().rename("n").reset_index()
               .sort_values("n").drop_duplicates("building_id", keep="last").set_index("building_id")["zip"])
    fill = df["zip"].isna() & df["building_id"].map(bld_zip).notna()
    df.loc[fill, "zip"] = df.loc[fill, "building_id"].map(bld_zip)
    df.loc[fill, "zip_imputed"] = True
    print(f"Housing: ZIP imputed from building_id for {fill.sum():,} rows")
    df = impute_zip_from_coords(df, label="Housing")
    df = reconcile_borough(df, "Housing")
    before = len(df)
    df = df.dropna(subset=["zip"])
    log["after dropping unlocatable rows"] = len(df)
    print(f"Housing: dropped {before - len(df):,} rows with no recoverable ZIP ({(before - len(df)) / before:.2%})")

    # 5) derived fields
    u = df["urgency"].fillna("")
    df["urgency_group"] = np.select(
        [u.str.contains("IMMEDIATE"), u.str.contains("HAZARD"), u.str.contains("NON"), u.str.contains("EMERG")],
        ["IMMEDIATE EMERGENCY", "HAZARDOUS", "NON EMERGENCY", "EMERGENCY"], default="UNKNOWN")
    df["is_urgent"] = df["urgency_group"].isin(["IMMEDIATE EMERGENCY", "EMERGENCY", "HAZARDOUS"])
    df["is_dup_report"] = df.get("duplicate_flag", pd.Series("N", index=df.index)).fillna("N").str.startswith("Y")
    df["month"] = df["received_date"].dt.to_period("M").dt.to_timestamp()
    df["year"] = df["received_date"].dt.year

    # 6) repair time (resolution days) with domain + IQR outlier handling
    closed = df["problem_status"].fillna("").str.startswith("CLOSE")
    days = (df["problem_status_date"] - df["received_date"]).dt.total_seconds() / 86_400
    days = days.where(closed & days.ge(0))          # negative durations are impossible -> missing
    log_days = np.log1p(days)
    q1, q3 = log_days.quantile([0.25, 0.75])
    upper = np.expm1(q3 + 1.5 * (q3 - q1))
    df["resolution_days"] = days
    df["resolution_outlier"] = days > upper
    df["resolution_days_capped"] = days.clip(upper=365)
    print(f"Housing: repair-time IQR upper fence (log scale) = {upper:.0f} days; "
          f"{df['resolution_outlier'].mean():.2%} of problems flagged; values capped at 365 days for averages")

    keep = ["problem_id", "complaint_id", "building_id", "borough", "zip", "zip_imputed", "received_date", "month", "year",
            "major_category", "minor_category", "urgency_group", "is_urgent", "is_dup_report", "problem_status",
            "resolution_days", "resolution_days_capped", "resolution_outlier", "latitude", "longitude"]
    df = df[[c for c in keep if c in df.columns]].reset_index(drop=True)
    for c in ["borough", "zip", "major_category", "minor_category", "urgency_group", "problem_status"]:
        if c in df:
            df[c] = df[c].astype("category")   # memory: millions of repeated strings -> integer codes
    print("Row log:", log)
    return df

housing = clean_housing(housing_raw)
import gc; gc.collect()
housing.info(memory_usage="deep")
housing.head()

Housing: ZIP imputed from building_id for 1,146 rows
Housing: ZIP imputed from coordinates for 333 of 355 candidates (median neighbour distance 22 m)
Housing: borough filled for 0 rows; 3,784 rows (0.23%) contradicted their ZIP and were corrected
Housing: dropped 43 rows with no recoverable ZIP (0.00%)
Housing: repair-time IQR upper fence (log scale) = 155 days; 1.06% of problems flagged; values capped at 365 days for averages
Row log: {'raw rows': 1679449, 'after exact duplicates': 1679449, 'after duplicate problem_id': 1679449, 'after date window': 1679449, 'after dropping unlocatable rows': 1679406}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1679406 entries, 0 to 1679405
Data columns (total 20 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   problem_id              1679406 non-null  object        
 1   complaint_id            1679406 non-null  object        
 2   building_id             

,problem_id,complaint_id,building_id,borough,zip,zip_imputed,received_date,month,year,major_category,minor_category,urgency_group,is_urgent,is_dup_report,problem_status,resolution_days,resolution_days_capped,resolution_outlier,latitude,longitude
0,24964474,12624398,5873,MANHATTAN,10033,False,2024-01-01 03:42:28,2024-01-01,2024,GENERAL,SIGNAGE MISSING,NON EMERGENCY,False,False,CLOSE,0.000579,0.000579,False,40.850951,-73.928072
1,24964644,12624540,429399,QUEENS,11103,False,2024-01-01 09:27:35,2024-01-01,2024,GENERAL,SIGNAGE MISSING,NON EMERGENCY,False,False,CLOSE,0.001505,0.001505,False,40.757347,-73.919225
2,24964593,12624503,807000,BRONX,10461,False,2024-01-01 08:44:57,2024-01-01,2024,HEAT/HOT WATER,ENTIRE BUILDING,EMERGENCY,True,True,CLOSE,0.050775,0.050775,False,40.839411,-73.845273
3,24964484,12624408,297808,BROOKLYN,11233,False,2024-01-01 04:43:20,2024-01-01,2024,HEAT/HOT WATER,APARTMENT ONLY,EMERGENCY,True,True,CLOSE,0.225417,0.225417,False,40.678242,-73.91029
4,24964459,12624386,233910,BROOKLYN,11226,False,2024-01-01 03:08:49,2024-01-01,2024,HEAT/HOT WATER,ENTIRE BUILDING,EMERGENCY,True,True,CLOSE,0.291296,0.291296,False,40.639768,-73.963006


In [12]:
# Validate the cleaning: repair-time distribution before/after capping, and category consolidation
fig = go.Figure()
fig.add_histogram(x=np.log1p(housing["resolution_days"].dropna()), nbinsx=60, name="all closed problems",
                  marker_color=PALETTE[0], opacity=0.9)
fig.add_vline(x=np.log1p(365), line_width=2, line_color=PALETTE[7], annotation_text="365-day cap")
style(fig, "Validation: repair time is right-skewed; cap applied only for averages",
      x="log(1 + days to close)", y="problems").update_layout(showlegend=False).show()
print(housing["urgency_group"].value_counts(normalize=True).round(3).to_string())
print(f"\nDuplicate-report share: {housing['is_dup_report'].mean():.1%}  |  ZIP imputed share: {housing['zip_imputed'].mean():.2%}")

urgency_group
EMERGENCY              0.613
NON EMERGENCY          0.349
IMMEDIATE EMERGENCY    0.037

Duplicate-report share: 16.1%  |  ZIP imputed share: 0.09%


### 4.2 Crashes
| Issue | Decision | Why |
|---|---|---|
| Repeated `collision_id` | Drop | A crash must count once |
| `crash_time` as text | Parse to hour; combine with date | Needed for the time-of-day question (RQ8) |
| Coordinates at 0 or outside NYC | Set to missing | Geocoding failures |
| Missing ZIP (large share) | **Nearest-neighbour imputation from coordinates** (≤200 m) | Dropping would bias against highways and arterials, which are the most dangerous roads |
| Missing injury counts | Rebuild from victim sub-counts; otherwise 0 | NYPD records each victim type; a missing total with known parts is a bookkeeping gap |
| Total < pedestrian + cyclist + motorist | Set total to the sum of parts, flag | Internal consistency rule |
| Injury outliers | IQR shown to fail on zero-inflated counts; use a **domain rule** (>20 injured = mass-casualty event) to *flag*, not drop | Bus crashes with 20+ injured are real and are exactly what a safety analysis must keep |
| Vehicle type free text | Map hundreds of spellings to 6 groups; crash takes the most consequential group involved | Makes the transport category filter meaningful |
| Contributing factor case / typos | Normalize case and known typos (`Illnes`); fall back to vehicle 2 when vehicle 1 is `Unspecified` | Consistent categories |

In [13]:
VEHICLE_RULES = [  # (group, keywords) checked in order; the first match wins
    ("Car / SUV", ["pick-up", "pickup", "pick up"]),   # pickups are mostly personal vehicles; checked before "truck"
    ("Bus", ["bus"]),
    ("Truck / Van", ["truck", "tractor", "trlr", "trail", "tanker", "dump", "box", "flat", "garbage", "refuse", "concrete",
                     "cement", "tow", "van", "delv", "deliv", "usps", "ups", "fedex", "semi", "chassis"]),
    ("Two-wheeler / Micromobility", ["bike", "bicycle", "e-b", "ebike", "scoot", "moped", "motorcycle", "motorbike",
                                     "minibike", "mini bike", "vespa", "e-sco", "unicycle"]),
    ("Taxi / For-hire", ["taxi", "cab", "livery", "limo", "for hire", "fhv", "uber"]),
    ("Car / SUV", ["sedan", "wagon", "sport utility", "suv", "pick", "convertible", "4 dr", "2 dr", "passenger", "car", "jeep",
                   "coupe", "hatch", "mini"]),
]
VEHICLE_PRIORITY = ["Bus", "Truck / Van", "Two-wheeler / Micromobility", "Taxi / For-hire", "Car / SUV", "Other / Unknown"]

def vehicle_group(raw_value) -> str:
    v = str(raw_value).lower()
    if v in ("", "nan", "none", "<na>", "unknown", "unk"):
        return "Other / Unknown"
    for group, keys in VEHICLE_RULES:
        if any(k in v for k in keys):
            return group
    return "Other / Unknown"

FACTOR_FIXES = {"illnes": "illness", "drugs (illegal)": "drugs (illegal)", "cell phone (hand-held)": "cell phone (hand-held)",
                "reaction to other uninvolved vehicle": "reaction to uninvolved vehicle"}

def std_factor(s: pd.Series) -> pd.Series:
    f = s.astype("string").str.strip().str.lower().replace(FACTOR_FIXES)
    f = f.mask(f.isin(["", "nan", "1", "80"]))  # numeric codes are legacy artefacts with no label
    return f.str.capitalize()

def clean_crashes(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    log = {"raw rows": len(df)}
    df = df.drop_duplicates("collision_id")
    log["after duplicate collision_id"] = len(df)

    # dates and times
    df["crash_date"] = pd.to_datetime(df["crash_date"].astype("string"), errors="coerce")
    t = pd.to_datetime(df["crash_time"].astype("string").str.strip(), format="%H:%M", errors="coerce")
    df["hour"] = t.dt.hour
    df = df[df["crash_date"].between(STUDY_START, pd.Timestamp(STUDY_END) + pd.Timedelta(days=1), inclusive="left")].copy()
    log["after date window"] = len(df)

    # geography
    df["borough"] = std_borough(df["borough"])
    df["zip"] = std_zip(df["zip"])
    df = clean_coords(df)
    df["zip_imputed"] = False
    df = impute_zip_from_coords(df, label="Crashes")
    df = reconcile_borough(df, "Crashes")
    n_before = len(df)
    df = df.dropna(subset=["zip"])
    log["after dropping unlocatable rows"] = len(df)
    print(f"Crashes: {n_before - len(df):,} crashes ({(n_before - len(df)) / n_before:.1%}) had neither ZIP nor usable coordinates; "
          "they stay out of ZIP-level analysis (reported as a limitation)")

    # victim counts: numeric, non-negative, internally consistent
    count_cols = [c for c in ["persons_injured", "persons_killed", "ped_injured", "ped_killed", "cyc_injured", "cyc_killed",
                              "mot_injured", "mot_killed"] if c in df]
    for c in count_cols:
        x = pd.to_numeric(df[c].astype("string"), errors="coerce")
        df[c] = x.where(x >= 0)
    for total, parts in [("persons_injured", ["ped_injured", "cyc_injured", "mot_injured"]),
                         ("persons_killed", ["ped_killed", "cyc_killed", "mot_killed"])]:
        parts = [p for p in parts if p in df]
        part_sum = df[parts].sum(axis=1, min_count=1)
        df[total] = df[total].fillna(part_sum)
        inconsistent = df[total] < part_sum
        df[f"{total}_fixed"] = inconsistent
        df.loc[inconsistent, total] = part_sum[inconsistent]
        print(f"Crashes: {inconsistent.sum():,} rows had {total} < sum of victim types (corrected)")
    df[count_cols] = df[count_cols].fillna(0).astype("int16")

    # outliers: show why IQR fails on zero-inflated counts, then apply a domain rule
    q1, q3 = df["persons_injured"].quantile([0.25, 0.75])
    iqr_fence = q3 + 1.5 * (q3 - q1)
    print(f"Crashes: IQR fence for injuries = {iqr_fence:.1f} -> would flag {(df['persons_injured'] > iqr_fence).mean():.1%} "
          f"of crashes (every crash with {int(np.floor(iqr_fence)) + 1}+ injured). Those are ordinary multi-occupant crashes, "
          "not errors, so IQR is the wrong tool for zero-inflated counts.")
    df["mass_casualty"] = df["persons_injured"] > 20
    print(f"Crashes: domain rule (>20 injured) flags {df['mass_casualty'].sum():,} crashes; kept as real events")

    # categories
    vt_cols = [c for c in ["vehicle_type_1", "vehicle_type_2"] if c in df]
    lookup = {v: vehicle_group(v) for v in pd.unique(df[vt_cols].astype("string").to_numpy().ravel())}
    rank = {g: i for i, g in enumerate(VEHICLE_PRIORITY)}
    ranks = np.column_stack([df[c].astype("string").map(lookup).map(rank).fillna(len(VEHICLE_PRIORITY) - 1) for c in vt_cols])
    df["vehicle_group"] = pd.Categorical([VEHICLE_PRIORITY[int(r)] for r in ranks.min(axis=1)], categories=VEHICLE_PRIORITY)
    f1 = std_factor(df["factor_1"])
    f2 = std_factor(df["factor_2"]) if "factor_2" in df else f1
    df["factor"] = f1.mask(f1.eq("Unspecified") & f2.notna() & f2.ne("Unspecified"), f2).fillna("Unspecified")

    # derived measures
    df["vru_casualties"] = df[["ped_injured", "ped_killed", "cyc_injured", "cyc_killed"]].sum(axis=1)
    df["is_injury_crash"] = df["persons_injured"] > 0
    df["is_night"] = df["hour"].isin([20, 21, 22, 23, 0, 1, 2, 3, 4, 5])
    df["month"] = df["crash_date"].dt.to_period("M").dt.to_timestamp()
    df["year"] = df["crash_date"].dt.year

    keep = ["collision_id", "crash_date", "month", "year", "hour", "is_night", "borough", "zip", "zip_imputed", "latitude",
            "longitude", "on_street_name", *count_cols, "vru_casualties", "is_injury_crash", "mass_casualty", "vehicle_group", "factor"]
    df = df[[c for c in keep if c in df.columns]].reset_index(drop=True)
    for c in ["borough", "zip", "factor"]:
        df[c] = df[c].astype("category")
    print("Row log:", log)
    return df

crashes = clean_crashes(crash_raw)
crashes.info(memory_usage="deep")
crashes.head()

Crashes: ZIP imputed from coordinates for 31,308 of 35,901 candidates (median neighbour distance 46 m)
Crashes: borough filled for 22,541 rows; 80 rows (0.05%) contradicted their ZIP and were corrected
Crashes: 11,708 crashes (6.6%) had neither ZIP nor usable coordinates; they stay out of ZIP-level analysis (reported as a limitation)
Crashes: 0 rows had persons_injured < sum of victim types (corrected)
Crashes: 0 rows had persons_killed < sum of victim types (corrected)
Crashes: IQR fence for injuries = 2.5 -> would flag 3.2% of crashes (every crash with 3+ injured). Those are ordinary multi-occupant crashes, not errors, so IQR is the wrong tool for zero-inflated counts.
Crashes: domain rule (>20 injured) flags 0 crashes; kept as real events
Row log: {'raw rows': 176875, 'after duplicate collision_id': 176875, 'after date window': 176875, 'after dropping unlocatable rows': 165167}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165167 entries, 0 to 165166
Data columns (total 25 colum

,collision_id,crash_date,month,year,hour,is_night,borough,zip,zip_imputed,latitude,longitude,on_street_name,persons_injured,persons_killed,ped_injured,ped_killed,cyc_injured,cyc_killed,mot_injured,mot_killed,vru_casualties,is_injury_crash,mass_casualty,vehicle_group,factor
0,697d463878a120792bd51ccd43f188d4,2024-01-07,2024-01-01,2024,11,False,MANHATTAN,10010,False,40.737545,-73.98409,EAST 21 STREET,1,0,1,0,0,0,0,0,1,True,False,Other / Unknown,Driver inexperience
1,029c2933287e4a4d16968760e8139cac,2024-01-14,2024-01-01,2024,13,False,BROOKLYN,11207,False,40.67325,-73.886734,PITKIN AVENUE,1,0,1,0,0,0,0,0,1,True,False,Other / Unknown,Unsafe speed
2,ac05ea451869313298171cdf37e026a6,2024-01-10,2024-01-01,2024,20,True,BROOKLYN,11226,False,40.655422,-73.95007,NOSTRAND AVENUE,1,0,1,0,0,0,0,0,1,True,False,Other / Unknown,Unspecified
3,b1732c3db65d8f6e8209d61860ffd300,2024-01-02,2024-01-01,2024,19,False,STATEN ISLAND,10310,False,40.632526,-74.10854,DAVIS AVENUE,1,0,1,0,0,0,0,0,1,True,False,Other / Unknown,Unspecified
4,8c1b5d06594c9bfa22c394b5ee3da896,2024-01-09,2024-01-01,2024,16,False,BRONX,10467,False,40.87319,-73.87541,NaN,1,0,1,0,0,0,0,0,1,True,False,Other / Unknown,Driver inattention/distraction


In [14]:
# Validate: vehicle grouping coverage and the effect of ZIP imputation
vg = crashes["vehicle_group"].value_counts(normalize=True).mul(100).round(1)
fig = px.bar(vg[::-1], orientation="h", color_discrete_sequence=[PALETTE[1]])
style(fig, "Validation: hundreds of raw vehicle strings consolidated into 6 groups", x="% of crashes", y=None
      ).update_layout(showlegend=False).show()
print(f"Crash ZIPs recovered by coordinate imputation: {crashes['zip_imputed'].mean():.1%} of the cleaned crashes")
print("Top factors after standardization:"); print(crashes["factor"].value_counts().head(10).to_string())

Crash ZIPs recovered by coordinate imputation: 19.0% of the cleaned crashes
Top factors after standardization:
factor
Does not apply                    36367
Driver inattention/distraction    32647
Unspecified                       27993
Failure to yield right-of-way     10150
Following too closely              6579
Other vehicular                    5536
Passing or lane usage improper     5094
Unsafe speed                       4684
Passing too closely                4245
Backing unsafely                   4045


## 5. Integration: the ZIP × month panel
**Join key:** `(zip, month)`. **Granularity justification:** ZIP is the finest unit both agencies report; month preserves seasonality
and gives stable counts. Both sides are aggregated to that grain first (many-to-one), so the join is one-to-one by construction and
we can **assert** key uniqueness.

**Metrics designed to avoid a spurious "big ZIP" correlation.** Raw counts in both datasets grow with population density, so a naive
scatter would "discover" that bigger places have more of everything. We therefore define two *intensity* indices with **different
denominators**, so no shared denominator can manufacture a correlation:
* **Housing Distress Index (HDI)** = urgent problems (emergency, immediate emergency, hazardous) **per complaining building**.
* **Street Danger Index (SDI)** = pedestrian and cyclist casualties (injured + killed) **per 100 crashes**: how often a crash hurts someone outside a car.

In [15]:
def aggregate_housing(h: pd.DataFrame, keys) -> pd.DataFrame:
    real = h[~h["is_dup_report"]]  # duplicate reports excluded from distress counts
    out = real.groupby(keys, observed=True).agg(
        problems=("problem_id", "size"),
        urgent=("is_urgent", "sum"),
        heat_problems=("major_category", lambda s: (s == "HEAT/HOT WATER").sum()),
        buildings=("building_id", "nunique"),
        resolved_n=("resolution_days_capped", "count"),
        resolved_days_sum=("resolution_days_capped", "sum"),
    ).reset_index()
    return out

def aggregate_crashes(c: pd.DataFrame, keys) -> pd.DataFrame:
    return c.groupby(keys, observed=True).agg(
        crashes=("collision_id", "size"),
        injury_crashes=("is_injury_crash", "sum"),
        injured=("persons_injured", "sum"),
        killed=("persons_killed", "sum"),
        ped_injured=("ped_injured", "sum"),
        cyc_injured=("cyc_injured", "sum"),
        vru_casualties=("vru_casualties", "sum"),
        night_crashes=("is_night", "sum"),
    ).reset_index()

h_zm = aggregate_housing(housing, ["zip", "month"])
c_zm = aggregate_crashes(crashes, ["zip", "month"])
for name, t in [("housing", h_zm), ("crash", c_zm)]:
    t["zip"] = t["zip"].astype(str)
    assert not t.duplicated(["zip", "month"]).any(), f"{name} aggregate has duplicate keys"

# Each side keeps its own borough so the post-join step can reconcile them
h_boro = housing.groupby("zip", observed=True)["borough"].agg(lambda s: s.mode().iat[0]).astype(str).rename("borough_housing")
c_boro = crashes.groupby("zip", observed=True)["borough"].agg(lambda s: s.mode().iat[0]).astype(str).rename("borough_crash")
h_zm = h_zm.merge(h_boro, left_on="zip", right_index=True, how="left")
c_zm = c_zm.merge(c_boro, left_on="zip", right_index=True, how="left")

panel_raw = h_zm.merge(c_zm, on=["zip", "month"], how="outer", indicator=True, validate="one_to_one")
print(panel_raw["_merge"].value_counts().to_string())
print(f"Panel: {len(panel_raw):,} ZIP-months, {panel_raw['zip'].nunique()} ZIPs, {panel_raw['month'].nunique()} months")

_merge
both          4161
right_only     264
left_only      118
Panel: 4,543 ZIP-months, 225 ZIPs, 24 months


The outer join is deliberate: a ZIP-month with crashes but **no** housing problems (or the reverse) is information, not noise.
`left_only` / `right_only` rows are exactly the new missing values the next section must resolve.

## 6. Post-integration cleaning
| Problem created by the join | Resolution | Justification |
|---|---|---|
| New NaNs in count columns (one side had no events) | Fill with **0** | Both datasets are complete registers of reported events: no record means zero reported events |
| NaNs in rates (denominator 0) | Leave as NaN | A rate with no exposure is undefined, not zero |
| Redundant `borough_housing` / `borough_crash` | Merge into one `borough`, report conflicts | One authoritative attribute |
| Type mismatches | `zip` -> string, `month` -> datetime, counts -> int | Prevents silent join and plotting errors |
| Non-residential / tiny ZIPs (airports, parks, PO boxes) | Drop ZIPs with < 100 housing problems **or** < 50 crashes over 2 years | Rates on a handful of events are unstable and not about neighborhoods |
| Incomplete grid | Reindex to the full ZIP × month grid | Time-series questions need every month present |
| Anomalous ZIP-months | Flag with IQR on log crash counts within each ZIP | Catches geocoding glitches without deleting real spikes |

In [16]:
COUNT_COLS = ["problems", "urgent", "heat_problems", "buildings", "resolved_n", "resolved_days_sum",
              "crashes", "injury_crashes", "injured", "killed", "ped_injured", "cyc_injured", "vru_casualties", "night_crashes"]

def post_integration_clean(p: pd.DataFrame) -> pd.DataFrame:
    p = p.copy()
    print("Missing values created by the join:"); print(p[COUNT_COLS].isna().sum().to_string())

    # redundant borough columns -> one
    conflict = p["borough_housing"].notna() & p["borough_crash"].notna() & (p["borough_housing"] != p["borough_crash"])
    print(f"\nZIP-months where the two datasets disagree on borough: {conflict.sum()} "
          f"({p.loc[conflict, 'zip'].nunique()} ZIPs; housing value kept because addresses are geocoded to the building)")
    p["borough"] = p["borough_housing"].fillna(p["borough_crash"])
    p = p.drop(columns=["borough_housing", "borough_crash", "_merge"])

    # full grid so every ZIP has every month
    grid = pd.MultiIndex.from_product([p["zip"].unique(), month_starts(STUDY_START, STUDY_END)], names=["zip", "month"])
    boro = p.dropna(subset=["borough"]).groupby("zip")["borough"].first()
    p = p.set_index(["zip", "month"]).reindex(grid).reset_index()
    p["borough"] = p["zip"].map(boro)

    # counts: missing -> 0, consistent integer types
    p[COUNT_COLS] = p[COUNT_COLS].fillna(0).astype("int64")
    p["zip"] = p["zip"].astype(str)
    p["month"] = pd.to_datetime(p["month"])
    p["year"] = p["month"].dt.year

    # drop non-residential / tiny ZIPs
    tot = p.groupby("zip")[["problems", "crashes"]].sum()
    keep = tot[(tot["problems"] >= 100) & (tot["crashes"] >= 50)].index
    print(f"Dropped {p['zip'].nunique() - len(keep)} low-volume ZIPs; {len(keep)} residential ZIPs remain")
    p = p[p["zip"].isin(keep)].copy()

    # anomaly flag: IQR on log crashes within each ZIP (kept, only flagged)
    lc = np.log1p(p["crashes"])
    g = lc.groupby(p["zip"])
    q1, q3 = g.transform(lambda s: s.quantile(0.25)), g.transform(lambda s: s.quantile(0.75))
    p["crash_month_anomaly"] = (lc > q3 + 1.5 * (q3 - q1)) | (lc < q1 - 1.5 * (q3 - q1))
    print(f"Flagged {p['crash_month_anomaly'].sum()} anomalous ZIP-months (kept, flagged)")

    assert p["borough"].notna().all(), "every ZIP must have a borough after reconciliation"
    assert not p.duplicated(["zip", "month"]).any()
    return p.reset_index(drop=True)

panel = post_integration_clean(panel_raw)
panel.dtypes.to_frame("dtype").T

Missing values created by the join:
problems             264
urgent               264
heat_problems        264
buildings            264
resolved_n           264
resolved_days_sum    264
crashes              118
injury_crashes       118
injured              118
killed               118
ped_injured          118
cyc_injured          118
vru_casualties       118
night_crashes        118

ZIP-months where the two datasets disagree on borough: 0 (0 ZIPs; housing value kept because addresses are geocoded to the building)
Dropped 54 low-volume ZIPs; 171 residential ZIPs remain
Flagged 96 anomalous ZIP-months (kept, flagged)


,zip,month,problems,urgent,heat_problems,buildings,resolved_n,resolved_days_sum,crashes,injury_crashes,injured,killed,ped_injured,cyc_injured,vru_casualties,night_crashes,borough,year,crash_month_anomaly
dtype,object,datetime64[ns],int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,object,int32,bool


In [17]:
def zip_summary(p: pd.DataFrame, buildings_by_zip: pd.Series) -> pd.DataFrame:
    """ZIP-level table with the two indices. Building stock = distinct buildings with any problem over the full window."""
    z = p.groupby(["zip", "borough"], as_index=False)[COUNT_COLS].sum().drop(columns="buildings")
    z["buildings"] = z["zip"].map(buildings_by_zip).astype(float)
    z["hdi"] = z["urgent"] / z["buildings"]                      # Housing Distress Index
    z["problems_per_bldg"] = z["problems"] / z["buildings"]
    z["urgent_share"] = z["urgent"] / z["problems"]
    z["heat_share"] = z["heat_problems"] / z["problems"]
    z["mean_repair_days"] = z["resolved_days_sum"] / z["resolved_n"].replace(0, np.nan)
    z["sdi"] = 100 * z["vru_casualties"] / z["crashes"]          # Street Danger Index
    z["injuries_per_100_crashes"] = 100 * z["injured"] / z["crashes"]
    z["night_share"] = z["night_crashes"] / z["crashes"]
    return z

buildings_by_zip = housing[~housing["is_dup_report"]].groupby("zip", observed=True)["building_id"].nunique()
buildings_by_zip.index = buildings_by_zip.index.astype(str)
Z = zip_summary(panel, buildings_by_zip)
Z = Z.merge(pd.concat([housing[["zip", "latitude", "longitude"]], crashes[["zip", "latitude", "longitude"]]])
            .assign(zip=lambda d: d["zip"].astype(str)).groupby("zip")[["latitude", "longitude"]].median(),
            left_on="zip", right_index=True, how="left")
Z.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
problems,171.0,8207.204678,9732.996886,103.0,1812.5,4320.0,10685.5,47351.0
urgent,171.0,4789.076023,5586.379879,68.0,1086.5,2568.0,6192.5,26266.0
heat_problems,171.0,1909.80117,2136.129531,28.0,484.0,1039.0,2407.0,10558.0
resolved_n,171.0,8198.760234,9722.804523,103.0,1812.5,4320.0,10671.5,47286.0
resolved_days_sum,171.0,135311.526316,148007.61601,1619.0,29393.5,72595.0,187670.5,597407.0
crashes,171.0,960.964912,544.761954,52.0,543.5,888.0,1255.5,3448.0
injury_crashes,171.0,419.076023,251.546113,25.0,222.0,377.0,553.5,1515.0
injured,171.0,558.643275,350.548639,35.0,290.5,497.0,764.0,2209.0
killed,171.0,2.54386,1.965019,0.0,1.0,2.0,4.0,10.0
ped_injured,171.0,104.74269,67.33842,7.0,51.5,92.0,154.0,343.0


In [18]:
# Validation plot: distribution of the two indices by borough (do the indices behave sensibly?)
fig = go.Figure()
for i, b in enumerate(BORO_ORDER):
    sub = Z[Z["borough"] == b]
    fig.add_box(y=sub["hdi"], name=b.title(), marker_color=BORO_COLOR[b], boxpoints="all", jitter=0.4, pointpos=0,
                hovertext=sub["zip"], line=dict(width=1.5))
style(fig, "Validation: Housing Distress Index by borough (each dot is a ZIP)", y="urgent problems per complaining building"
      ).update_layout(showlegend=False).show()

## 7. Research questions
Ten questions, two per team member, each answered by one visualization. Helper functions below produce correlation statistics and
a plain-language verdict so the interpretation is computed from the data rather than written in advance.

**Statistics used.** Spearman's rank correlation (robust to skew and outliers, no linearity assumption) and a **partial Spearman**
that controls for building stock (log buildings) so a link is not just "bigger ZIPs have more of everything".

In [19]:
def strength(r):
    a = abs(r)
    return "negligible" if a < 0.1 else "weak" if a < 0.3 else "moderate" if a < 0.5 else "strong"

def partial_spearman(x, y, z):
    """Spearman correlation of x and y after removing the linear effect of z from the ranks of both."""
    d = pd.DataFrame({"x": x, "y": y, "z": z}).dropna().rank()
    rx = d["x"] - np.polyval(np.polyfit(d["z"], d["x"], 1), d["z"])
    ry = d["y"] - np.polyval(np.polyfit(d["z"], d["y"], 1), d["z"])
    r = np.corrcoef(rx, ry)[0, 1]
    n = len(d)
    t = r * np.sqrt((n - 3) / max(1e-12, 1 - r ** 2))
    p = 2 * stats.t.sf(abs(t), n - 3)
    return r, p, n

def verdict(label, x, y, control=None):
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    r, p = stats.spearmanr(d["x"], d["y"])
    msg = f"{label}: Spearman rho = {r:.2f} ({strength(r)}, p = {p:.3g}, n = {len(d)} ZIPs)"
    if control is not None:
        pr, pp, _ = partial_spearman(x, y, control)
        msg += f" | controlling for building stock: rho = {pr:.2f} ({strength(pr)}, p = {pp:.3g})"
    print(msg)
    return r, p

def quartile(s):
    return pd.qcut(s.rank(method="first"), 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])

Z["log_buildings"] = np.log(Z["buildings"])
Z["hdi_q"] = quartile(Z["hdi"])
Z["sdi_q"] = quartile(Z["sdi"])

---
### Member 1: The geography of overlap
**RQ1. Do ZIP codes with more housing-maintenance problems also record more traffic injuries, and does the link survive once we control
for how many residential buildings the ZIP has?**
*Chart:* log-log scatter, one point per ZIP, borough by colour **and** marker shape. *Why:* both counts span orders of magnitude, so a log
scale shows the whole range; the partial correlation tests whether "size" explains the pattern.

In [20]:
fig = px.scatter(Z, x="problems", y="injured", color="borough", symbol="borough", log_x=True, log_y=True,
                 color_discrete_map=BORO_COLOR, symbol_map=BORO_SYMBOL, category_orders={"borough": BORO_ORDER},
                 hover_data={"zip": True, "buildings": ":,", "problems": ":,", "injured": ":,"}, trendline="ols",
                 trendline_scope="overall", trendline_options=dict(log_x=True, log_y=True), trendline_color_override=GRAY)
fig.update_traces(marker=dict(size=9, line=dict(width=1, color="white")), selector=dict(mode="markers"))
style(fig, "RQ1. Housing problems vs traffic injuries per ZIP (2024 to 2025)", x="housing problems (log)", y="people injured in crashes (log)").show()
verdict("RQ1 problems vs injuries", Z["problems"], Z["injured"], control=Z["log_buildings"])

RQ1 problems vs injuries: Spearman rho = 0.62 (strong, p = 1.4e-19, n = 171 ZIPs) | controlling for building stock: rho = -0.11 (weak, p = 0.158)


(np.float64(0.620540648238152), np.float64(1.4007034678747462e-19))

**Reading RQ1.** The raw association is expected to be positive simply because large ZIPs have more of everything. The number to judge is the
*partial* rho: if it stays clearly above zero after controlling for building stock, housing distress and injuries co-locate beyond size;
if it collapses toward zero, the raw pattern was mostly a size effect. Both outcomes are reported honestly on the website.

**RQ2. Where are the "double-burden" ZIPs, the ones in the top quartile of both the Housing Distress Index and the Street Danger Index?**
*Chart:* map of ZIP centroids coloured by burden quadrant, sized by people injured. *Why:* the answer is a place, so it needs a map.

In [21]:
def burden_class(z):
    hi_h, hi_s = z["hdi"] >= z["hdi"].quantile(0.75), z["sdi"] >= z["sdi"].quantile(0.75)
    return np.select([hi_h & hi_s, hi_h, hi_s], ["Double burden", "Housing distress only", "Street danger only"], "Neither (below Q4)")

BURDEN_COLORS = {"Double burden": PALETTE[7], "Housing distress only": PALETTE[1], "Street danger only": PALETTE[0], "Neither (below Q4)": GRAY}
Z["burden"] = burden_class(Z)
HAS_MAPLIBRE = hasattr(px, "scatter_map")          # plotly >= 5.24 uses MapLibre (no token); older versions use mapbox
scatter_map = px.scatter_map if HAS_MAPLIBRE else px.scatter_mapbox
fig = scatter_map(Z, lat="latitude", lon="longitude", color="burden", size="injured", size_max=22, zoom=9.3,
                  center=dict(lat=40.70, lon=-73.94), color_discrete_map=BURDEN_COLORS,
                  category_orders={"burden": list(BURDEN_COLORS)}, hover_name="zip",
                  hover_data={"borough": True, "hdi": ":.2f", "sdi": ":.1f", "injured": ":,", "latitude": False, "longitude": False})
fig.update_layout(**({"map_style": "carto-positron"} if HAS_MAPLIBRE else {"mapbox_style": "carto-positron"}))
style(fig, "RQ2. Double-burden ZIPs: top quartile for housing distress AND street danger", height=620).show()
double = Z[Z["burden"] == "Double burden"].sort_values("hdi", ascending=False)
expected = len(Z) / 16
print(f"Double-burden ZIPs: {len(double)} (if the two indices were independent we would expect about {expected:.0f})")
print(double[["zip", "borough", "hdi", "sdi", "injured"]].round(2).to_string(index=False))

Double-burden ZIPs: 10 (if the two indices were independent we would expect about 11)
  zip   borough   hdi   sdi  injured
10037 MANHATTAN 25.02 22.14      278
10456     BRONX 20.39 24.52      715
11226  BROOKLYN 18.59 21.23     1146
10030 MANHATTAN 18.00 23.75      265
10455     BRONX 17.69 20.43      664
10026 MANHATTAN 16.80 23.22      236
10027 MANHATTAN 13.57 20.98      675
11372    QUEENS 12.44 22.46      656
11230  BROOKLYN 12.38 21.80      783
10029 MANHATTAN 11.76 20.87      935


**Reading RQ2.** Under independence, one ZIP in sixteen would land in the double-burden corner by chance. A count well above that baseline
means the two burdens concentrate in the same neighborhoods; a count near it means they do not.

---
### Member 2: Time
**RQ3. Do housing problems and street injuries share a seasonal rhythm across the year?**
*Chart:* indexed line chart (each series' 24-month mean = 100) on **one** axis. *Why:* the series have very different magnitudes; indexing
avoids a misleading dual-axis chart.

In [22]:
city = panel.groupby("month")[["problems", "heat_problems", "injured"]].sum()
indexed = city / city.mean() * 100
fig = go.Figure()
for i, (col, name) in enumerate([("problems", "All housing problems"), ("heat_problems", "Heat / hot-water problems"),
                                 ("injured", "People injured in crashes")]):
    fig.add_scatter(x=indexed.index, y=indexed[col], name=name, mode="lines+markers", line=dict(width=2, color=PALETTE[i]),
                    marker=dict(size=6), hovertemplate="%{x|%b %Y}: %{y:.0f}<extra>" + name + "</extra>")
fig.add_hline(y=100, line_width=1, line_color=GRAY)
style(fig, "RQ3. Seasonal rhythm, indexed to each series' own average (= 100)", y="index (24-month mean = 100)").show()
r, p = stats.spearmanr(city["problems"], city["injured"])
print(f"RQ3 month-to-month citywide Spearman rho (problems vs injuries) = {r:.2f}, p = {p:.3g}, n = 24 months")
print("Peak month, heat problems:", indexed["heat_problems"].idxmax().strftime("%b %Y"),
      "| peak month, injuries:", indexed["injured"].idxmax().strftime("%b %Y"))

RQ3 month-to-month citywide Spearman rho (problems vs injuries) = -0.51, p = 0.0113, n = 24 months
Peak month, heat problems: Jan 2025 | peak month, injuries: Sep 2025


**RQ4. Beyond the shared calendar, do a ZIP's bad housing months coincide with its bad street months?**
We remove each ZIP's own average and the citywide month effect (two-way demeaning), then correlate the leftover monthly anomalies inside
each ZIP. *Chart:* histogram of the per-ZIP correlations. *Why:* a distribution centred on zero says the burdens co-locate but do not
co-move; a distribution shifted right would suggest shared short-term shocks.

In [23]:
def two_way_anomaly(p, col):
    """Residual after removing ZIP mean and month mean (on log1p counts)."""
    y = np.log1p(p[col])
    return y - y.groupby(p["zip"]).transform("mean") - y.groupby(p["month"]).transform("mean") + y.mean()

panel["h_anom"] = two_way_anomaly(panel, "problems")
panel["c_anom"] = two_way_anomaly(panel, "injured")
zip_corr = panel.groupby("zip")[["h_anom", "c_anom"]].apply(lambda g: g["h_anom"].corr(g["c_anom"], method="spearman")).dropna()
fig = px.histogram(zip_corr, nbins=30, color_discrete_sequence=[PALETTE[0]])
fig.add_vline(x=0, line_width=1, line_color=GRAY)
fig.add_vline(x=zip_corr.median(), line_width=2, line_color=PALETTE[7], annotation_text=f"median {zip_corr.median():.2f}")
style(fig, "RQ4. Within-ZIP co-movement of housing and injury anomalies (after removing seasonality)",
      x="Spearman rho between monthly anomalies (per ZIP)", y="number of ZIPs").update_layout(showlegend=False).show()
w = stats.wilcoxon(zip_corr)
print(f"RQ4 median within-ZIP rho = {zip_corr.median():.2f}; Wilcoxon test that the median is 0: p = {w.pvalue:.3g}; "
      f"{(zip_corr > 0).mean():.0%} of ZIPs positive")

RQ4 median within-ZIP rho = 0.01; Wilcoxon test that the median is 0: p = 0.443; 51% of ZIPs positive


---
### Member 3: What kind of distress
**RQ5. Which housing problem categories track pedestrian and cyclist casualties most closely across ZIPs (after controlling for building stock)?**
*Chart:* ranked horizontal bars of partial Spearman rho, one per major category. *Why:* ranking answers "which" directly.

In [24]:
cat_zip = (housing[~housing["is_dup_report"]].assign(zip=lambda d: d["zip"].astype(str))
           .groupby(["zip", "major_category"], observed=True).size().unstack(fill_value=0))
cat_zip = cat_zip.loc[cat_zip.index.isin(Z["zip"])]
top_cats = cat_zip.sum().sort_values(ascending=False).head(12).index
zi = Z.set_index("zip")
rows = []
for cat in top_cats:
    rate = cat_zip[cat] / zi.loc[cat_zip.index, "buildings"]          # category problems per building
    r, p, n = partial_spearman(rate, zi.loc[cat_zip.index, "sdi"], zi.loc[cat_zip.index, "log_buildings"])
    rows.append({"category": cat.title(), "partial_rho": r, "p": p})
rq5 = pd.DataFrame(rows).sort_values("partial_rho")
rq5["significant (p<0.05)"] = np.where(rq5["p"] < 0.05, "yes", "no")
fig = px.bar(rq5, x="partial_rho", y="category", orientation="h", color="significant (p<0.05)",
             color_discrete_map={"yes": PALETTE[0], "no": "#c3c2b7"}, hover_data={"p": ":.3g"})
fig.add_vline(x=0, line_width=1, line_color=GRAY)
style(fig, "RQ5. Which housing problems track the Street Danger Index? (per-building rates, size-controlled)",
      x="partial Spearman rho with SDI", y=None, height=520).show()
print(rq5.round(3).to_string(index=False))

            category  partial_rho     p significant (p<0.05)
            Electric       -0.098 0.205                   no
           Appliance        0.047 0.539                   no
       Paint/Plaster        0.108 0.160                   no
          Water Leak        0.111 0.151                   no
              Safety        0.135 0.079                   no
            Plumbing        0.136 0.077                   no
Unsanitary Condition        0.153 0.047                  yes
     Flooring/Stairs        0.161 0.035                  yes
         Door/Window        0.162 0.034                  yes
      Line Of Travel        0.238 0.002                  yes
             General        0.294 0.000                  yes
      Heat/Hot Water        0.303 0.000                  yes


**RQ6. Which crash causes are over-represented in high-distress ZIPs compared with low-distress ZIPs?**
*Chart:* diverging bars of *lift* = (factor share in top-HDI-quartile ZIPs ÷ share in bottom-quartile ZIPs) − 1. *Why:* a diverging
scale centred on zero separates "more common" (red) from "less common" (blue) at a glance.

In [25]:
cr = crashes.assign(zip=crashes["zip"].astype(str)).merge(Z[["zip", "hdi_q"]], on="zip", how="inner")
cr["factor"] = cr["factor"].astype(str)           # drop unused categories so counts only show real factors
cr = cr[cr["factor"] != "Unspecified"].copy()
share = (cr.groupby(["hdi_q", "factor"], observed=True).size() / cr.groupby("hdi_q", observed=True).size()).unstack(0)
top_factors = cr["factor"].value_counts().head(14).index
lift = (share.loc[top_factors, "Q4 (highest)"] / share.loc[top_factors, "Q1 (lowest)"] - 1).mul(100).sort_values()
fig = go.Figure(go.Bar(x=lift.values, y=lift.index, orientation="h",
                       marker_color=np.where(lift.values >= 0, PALETTE[7], PALETTE[0]),
                       hovertemplate="%{y}: %{x:+.0f}%<extra></extra>"))
fig.add_vline(x=0, line_width=1, line_color=GRAY)
style(fig, "RQ6. Crash causes over- (red) and under- (blue) represented in the most housing-distressed ZIPs",
      x="% difference in share, top vs bottom HDI quartile", y=None, height=540).show()
ct = pd.crosstab(cr["hdi_q"].isin(["Q4 (highest)"]), cr["factor"].isin(top_factors[:5]))
print(f"RQ6 chi-square test that factor mix is independent of distress tier: p = {stats.chi2_contingency(ct)[1]:.3g}")

RQ6 chi-square test that factor mix is independent of distress tier: p = 3.47e-11


---
### Member 4: Vulnerability and response
**RQ7. Do residents of ZIPs with more dangerous streets also wait longer for housing repairs?**
*Chart:* box plot of ZIP mean repair time by Street Danger Index quartile. *Why:* shows the full spread per tier, not just a mean.

In [26]:
fig = px.box(Z, x="sdi_q", y="mean_repair_days", points="all", color_discrete_sequence=[PALETTE[0]],
             category_orders={"sdi_q": ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]}, hover_data=["zip", "borough"])
style(fig, "RQ7. Housing repair time by Street Danger Index quartile", x="Street Danger Index quartile",
      y="mean days to close a problem (capped at 365)").update_layout(showlegend=False).show()
groups = [g["mean_repair_days"].dropna() for _, g in Z.groupby("sdi_q", observed=True)]
print(f"RQ7 Kruskal-Wallis across quartiles: p = {stats.kruskal(*groups).pvalue:.3g}")
verdict("RQ7 SDI vs repair days", Z["sdi"], Z["mean_repair_days"])

RQ7 Kruskal-Wallis across quartiles: p = 3.58e-06
RQ7 SDI vs repair days: Spearman rho = 0.39 (moderate, p = 1.93e-07, n = 171 ZIPs)


(np.float64(0.38544651642786865), np.float64(1.9297783636577104e-07))

**RQ8. Do crashes in the most housing-distressed ZIPs happen at different hours than in the least distressed ones?**
*Chart:* hour-of-day profile (% of each tier's crashes per hour), two lines. *Why:* the shape of the day, not totals, is the question;
percentages remove the size difference between tiers.

In [27]:
hr = cr.dropna(subset=["hour"])
prof = (hr.groupby(["hdi_q", "hour"], observed=True).size() / hr.groupby("hdi_q", observed=True).size()).mul(100).unstack(0)
fig = go.Figure()
for i, (q, name) in enumerate([("Q4 (highest)", "Most distressed ZIPs (HDI Q4)"), ("Q1 (lowest)", "Least distressed ZIPs (HDI Q1)")]):
    fig.add_scatter(x=prof.index, y=prof[q], name=name, mode="lines", line=dict(width=2, color=[PALETTE[7], PALETTE[0]][i]),
                    hovertemplate="%{x}:00 %{y:.1f}%<extra>" + name + "</extra>")
fig.add_vrect(x0=20, x1=23.5, fillcolor="#ecebe8", opacity=0.5, line_width=0)
fig.add_vrect(x0=-0.5, x1=5, fillcolor="#ecebe8", opacity=0.5, line_width=0, annotation_text="dark hours", annotation_position="top left")
style(fig, "RQ8. Hour-of-day crash profile: most vs least housing-distressed ZIPs", x="hour of day", y="% of the tier's crashes"
      ).update_xaxes(dtick=2).show()
night = hr.groupby(hr["hdi_q"].isin(["Q4 (highest)"]))["is_night"].mean()
print(f"RQ8 night share: high-distress ZIPs {night.get(True, np.nan):.1%} vs other ZIPs {night.get(False, np.nan):.1%}")

RQ8 night share: high-distress ZIPs 31.0% vs other ZIPs 30.3%


---
### Member 5: Change and composite stress
**RQ9. When and where does combined neighborhood stress peak?** A monthly *Combined Stress Score* per borough = mean of the z-scores of
urgent problems per building and pedestrian/cyclist casualties per 100 crashes. *Chart:* borough × month heatmap. *Why:* two categorical
axes and one magnitude, exactly what a heatmap encodes.

In [28]:
bm = panel.groupby(["borough", "month"], as_index=False)[["urgent", "buildings", "vru_casualties", "crashes"]].sum()
bm["hdi_m"] = bm["urgent"] / bm["buildings"].replace(0, np.nan)
bm["sdi_m"] = 100 * bm["vru_casualties"] / bm["crashes"].replace(0, np.nan)
for c in ["hdi_m", "sdi_m"]:
    bm[c + "_z"] = (bm[c] - bm[c].mean()) / bm[c].std()
bm["stress"] = bm[["hdi_m_z", "sdi_m_z"]].mean(axis=1)
heat = bm.pivot(index="borough", columns="month", values="stress").reindex(BORO_ORDER)
fig = go.Figure(go.Heatmap(z=heat.values, x=heat.columns, y=[b.title() for b in heat.index], zmid=0,
                           colorscale=[[0, "#2a78d6"], [0.5, "#f0efec"], [1, "#e34948"]],
                           colorbar=dict(title="stress<br>(z)"), hovertemplate="%{y}, %{x|%b %Y}: %{z:.2f}<extra></extra>"))
style(fig, "RQ9. Combined Stress Score by borough and month (red = above average)", height=380).show()
print("Highest-stress borough-month:", bm.loc[bm["stress"].idxmax(), ["borough", "month", "stress"]].to_dict())

Highest-stress borough-month: {'borough': 'BRONX', 'month': Timestamp('2025-01-01 00:00:00'), 'stress': 1.7085178576487468}


**RQ10. Between 2024 and 2025, did ZIPs where housing problems grew also see traffic injuries grow?**
*Chart:* scatter of % change in problems vs % change in injuries, with quadrant lines at zero. *Why:* each quadrant is a story (both
worse, both better, diverging).

In [29]:
yz = panel.groupby(["zip", "borough", "year"])[["problems", "injured"]].sum().unstack("year")
chg = pd.DataFrame({
    "problems_change_%": (yz[("problems", 2025)] / yz[("problems", 2024)] - 1) * 100,
    "injured_change_%": (yz[("injured", 2025)] / yz[("injured", 2024)].replace(0, np.nan) - 1) * 100,
}).replace([np.inf, -np.inf], np.nan).dropna().reset_index()
chg = chg[chg["problems_change_%"].between(-90, 300) & chg["injured_change_%"].between(-90, 300)]  # drop tiny-base explosions
fig = px.scatter(chg, x="problems_change_%", y="injured_change_%", color="borough", symbol="borough",
                 color_discrete_map=BORO_COLOR, symbol_map=BORO_SYMBOL, category_orders={"borough": BORO_ORDER}, hover_name="zip")
fig.update_traces(marker=dict(size=9, line=dict(width=1, color="white")))
fig.add_hline(y=0, line_width=1, line_color=GRAY); fig.add_vline(x=0, line_width=1, line_color=GRAY)
style(fig, "RQ10. Change 2024 to 2025: housing problems vs traffic injuries per ZIP", x="% change in housing problems",
      y="% change in people injured").show()
verdict("RQ10 change vs change", chg["problems_change_%"], chg["injured_change_%"])
up_h, up_c = chg["problems_change_%"] > 0, chg["injured_change_%"] > 0
print("Quadrant counts:", pd.Series(np.select([up_h & up_c, ~up_h & ~up_c], ["both up", "both down"], "diverging")).value_counts().to_dict())

RQ10 change vs change: Spearman rho = 0.12 (weak, p = 0.108, n = 171 ZIPs)
Quadrant counts: {'diverging': 73, 'both down': 54, 'both up': 44}


## 8. Export for the website
The website never touches the raw millions of rows. It reads small pre-aggregated tables (a few MB in total) so it loads fast on a free
Render instance. Each table keeps the filter dimensions the site needs: **borough, year/month, housing major category, vehicle group**.

In [32]:
zip_keep = set(Z["zip"])
hh = housing[~housing["is_dup_report"]].assign(zip=lambda d: d["zip"].astype(str))
hh = hh[hh["zip"].isin(zip_keep)]
cc = crashes.assign(zip=lambda d: d["zip"].astype(str))
cc = cc[cc["zip"].isin(zip_keep)]

exports = {
    "housing_zip_month_category.csv.gz": hh.groupby(["zip", "month", "major_category"], observed=True).agg(
        problems=("problem_id", "size"), urgent=("is_urgent", "sum"), resolved_n=("resolution_days_capped", "count"),
        resolved_days_sum=("resolution_days_capped", "sum")).reset_index(),
    "crash_zip_month_vehicle.csv.gz": cc.groupby(["zip", "month", "vehicle_group"], observed=True).agg(
        crashes=("collision_id", "size"), injured=("persons_injured", "sum"), killed=("persons_killed", "sum"),
        vru_casualties=("vru_casualties", "sum"), night_crashes=("is_night", "sum")).reset_index(),
    "crash_zip_year_vehicle_hour.csv.gz": cc.dropna(subset=["hour"]).groupby(["zip", "year", "vehicle_group", "hour"], observed=True)
        .size().rename("crashes").reset_index(),
    "crash_zip_year_vehicle_factor.csv.gz": cc.groupby(["zip", "year", "vehicle_group", "factor"], observed=True).agg(
        crashes=("collision_id", "size"), vru_casualties=("vru_casualties", "sum")).reset_index(),
    "zip_meta.csv": Z[["zip", "borough", "buildings", "latitude", "longitude"]],
}
for name, df in exports.items():
    df.to_csv(PROC_DIR / name, index=False)
    print(f"{name:<42} {len(df):>8,} rows  {(PROC_DIR / name).stat().st_size / 1e6:6.2f} MB")

housing_zip_month_category.csv.gz            44,278 rows    0.56 MB
crash_zip_month_vehicle.csv.gz               18,264 rows    0.10 MB
crash_zip_year_vehicle_hour.csv.gz           28,356 rows    0.10 MB
crash_zip_year_vehicle_factor.csv.gz         23,216 rows    0.11 MB
zip_meta.csv                                    171 rows    0.01 MB


In [33]:
# In Colab: download the processed folder and commit it to the repo's data/processed/ directory
try:
    import shutil
    from google.colab import files  # noqa: F401
    shutil.make_archive("processed", "zip", PROC_DIR)
    files.download("processed.zip")
except ImportError:
    print(f"Not on Colab: processed files are in {PROC_DIR.resolve()}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Story, limitations, impact

### Key findings (fill the numbers from the printed outputs above before submission)
* **Co-location (RQ1, RQ2):** see the partial rho and the double-burden count vs its random baseline.
* **Co-movement (RQ3, RQ4):** shared seasonality is expected (winter heat complaints vs summer crash peak); the within-ZIP anomaly test
  tells us whether the two burdens share short-term shocks.
* **Mechanism hints (RQ5, RQ6, RQ8):** which housing problems and which crash causes and hours distinguish the
  distressed ZIPs.
* **Response gap (RQ7):** whether repair times differ by street danger tier.

### Limitations (what the data cannot tell us)
1. **Correlation, not causation.** Both burdens plausibly share upstream causes (income, age of housing stock, arterial roads, policing and
   reporting behaviour). Nothing here shows that bad buildings cause crashes or the reverse.
2. **Complaints measure reporting, not conditions.** Tenants who fear retaliation or do not know about 311 report less; distress may be
   under-counted exactly where it is worst.
3. **Crashes measure police-reported crashes.** Minor crashes without a police report are missing, and crashes with neither a ZIP nor coordinates
   (share printed in Section 4.2) are excluded from ZIP-level analysis.
4. **Building stock proxy.** "Complaining buildings" is a proxy for residential stock; ZIPs with many commercial blocks (Midtown) can look
   extreme on per-building measures.
5. **ZIP codes are postal routes, not neighborhoods**, and the two-year window cannot show long-run trends.
6. **Ecological fallacy.** A ZIP-level association says nothing about any individual household.

### Impact: who could use this
* **HPD + DOT jointly:** prioritize double-burden ZIPs for coordinated proactive inspections and Vision Zero street redesigns.
* **Community boards and tenant groups:** evidence that a neighborhood carries two burdens at once strengthens funding requests.
* **Researchers:** the size-controlled indices and partial correlations are a template for avoiding "big places have more of everything".

## Team contributions
| Member | Research questions | Visualizations | Data cleaning | Integration | Website |
|---|---|---|---|---|---|
| Member 1 (Mohammed Walid, 13003322) | RQ1, RQ2 | log-log scatter, burden map | shared geo helpers (borough, ZIP, coordinates) | ZIP × month key design | map + cross-domain scatter |
| Member 2 (Abdelrahman Farajallah, 13003482) | RQ3, RQ4 | indexed line, anomaly histogram | date parsing, study window | full grid reindex (post-join) | time-series panels |
| Member 3 (Omar Turki, 19010638) | RQ5, RQ6 | ranked bars, diverging bars | housing category standardization | housing aggregation | category filter + search parser |
| Member 4 (Mostafa Eltatawi, 13007048) | RQ7, RQ8 | box plot, hour profile | repair-time outliers (IQR + cap) | post-join NaN policy | KPI tiles |
| Member 5 (Ahmed Raslan, 13005475) | RQ9, RQ10 | heatmap, change scatter | crash victim counts consistency | crash aggregation | Generate Report callback |



## Prompts used
* AI assistant (Claude) prompt: *"Go through the project details and make the project, choose whatever dataset you want to choose, make sure
  to go through the details line by line, and for this project you are a professional data engineer and visualizer."* Used for dataset
  selection, pipeline design, code scaffolding, website scaffolding and README drafting. All outputs were reviewed and run by the team.
* (Add every further prompt the team used here.)

## Dataset links
* Housing Maintenance Code Complaints and Problems: https://data.cityofnewyork.us/d/ygpa-z7cr
* Motor Vehicle Collisions, Crashes: https://data.cityofnewyork.us/d/h9gi-nx95